In [0]:
from pyspark.sql.functions import col

# =========================================================
# STEP-5 : GET RAW INVALID RECORDS FROM QUARANTINE
# =========================================================

invalid_df = spark.read.table("retails.silver.categories_quarantine") \
                    .filter((col("_rescued_data").isNotNull()) & (col("quarantine_status") == 'NEW'))

In [0]:
from pyspark.sql.functions import col, get_json_object as get_json_from

# =========================================================
# STEP-6 : TRY TO RECOVER RESCUED COLUMNS
# =========================================================
recovered_df = invalid_df \
                .withColumn("category_id_fixed", get_json_from(col("_rescued_data"), "$.category_id")) \
                .withColumn("category_department_id_fixed", get_json_from(col("_rescued_data"), "$.category_department_id")) \
                .withColumn("category_name_fixed", get_json_from(col("_rescued_data"), "$.category_name")) 


In [0]:
from pyspark.sql.functions import coalesce, trim

# =========================================================
# STEP-7 : MERGE RECOVERED VALUES
# =========================================================

recovered_df = recovered_df \
    .withColumn(
        "category_id",
        coalesce(col("category_id"), col("category_id_fixed").cast("bigint"))
    ) \
    .withColumn(
        "category_department_id",
        coalesce(col("category_department_id"), col("category_department_id_fixed").cast("bigint"))
    ) \
    .withColumn(
        "category_name",
        coalesce(col("category_name"), trim(col("category_name_fixed").cast("string")))
    ) 

In [0]:
recovered_df = recovered_df.drop("category_id_fixed", "category_name_fixed", "category_department_id_fixed")

In [0]:
from pyspark.sql.functions import col

# =========================================================
# STEP-8 : APPLY DATA QUALITY RULES
# =========================================================

cleaned_recovered_df = recovered_df.filter(
    col("category_id").isNotNull() &
    col("category_name").isNotNull() &
    col("category_department_id").isNotNull() &
    (col("category_department_id") >= 0)
)

In [0]:
cleaned_recovered_df = cleaned_recovered_df.dropDuplicates(["category_id"])
cleaned_recovered_df.createOrReplaceTempView("categories_cleaned_vw_fixed")

In [0]:
from pyspark.sql.functions import when, current_timestamp, sha2, concat_ws

cleaned_recovered_df = cleaned_recovered_df \
    .withColumn("category_id", col("category_id").cast("bigint")) \
    .withColumn("category_department_id", col("category_department_id").cast("bigint")) \
    .withColumn("batch_id", col("batch_id").cast("integer")) \
    .withColumn("is_deleted", when(col("op")=='DELETE', True).otherwise(False)) \
    .withColumn("category_name", when(col("category_name").isNull(), "Unknown").otherwise(col("category_name")))

cleaned_recovered_df = cleaned_recovered_df.select("category_id", "category_name", "category_department_id", "op", "is_deleted", "source_system", "source_file_name", "ingestion_ts", "ingestion_dt", "batch_id", "run_id")

cleaned_recovered_df = cleaned_recovered_df.withColumn("event_ts", current_timestamp()) \
                    .withColumn("record_hash",
                                sha2(
                                    concat_ws(
                                        "||",
                                        col("category_id"),
                                        col("category_name"),
                                        col("category_department_id")
                                    ),
                                    256
                                )
                            )
    

try:
    cleaned_recovered_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "false") \
        .saveAsTable("retails.silver.categories_cdc")

except Exception as e:
    print(str(e))

In [0]:
merge_query_rescued = """
    MERGE INTO retails.silver.categories_quarantine t
    USING categories_cleaned_vw_fixed s
    ON t.category_id = s.category_id
    WHEN MATCHED THEN
        UPDATE SET t.quarantine_status = 'FIXED', t.reprocessed_at = current_timestamp()
 
    """
spark.sql(merge_query_rescued).show()


In [0]:
update_query_corrupt = """
        UPDATE retails.silver.categories_quarantine
        SET
            quarantine_status = 'INVALID',
            reprocessed_at = current_timestamp()
        WHERE quarantine_status = 'NEW'
"""

spark.sql(update_query_corrupt).show()

In [0]:
%sql
-- select * from retails.silver.categories_quarantine;
-- select category_department_id from retails.silver.categories_cdc;

-- describe retails.silver.categories_quarantine;
